<table>
  <tr>
    <td style="text-align: center">
      <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download-url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
      </a>
    </td>
  </tr>
</table>
<br clear="all"/>

---

# Track 1 (Notebook 02): Agentic Medallion Transformation (`acsm_bronze` → `acsm_silver` → `acsm_gold` + Airflow Orchestration)
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

---

## 🗺️ Notebook 02 Architecture & Medallion DAG (`acsm_bronze` → `acsm_silver` → `acsm_gold`)

![Track 1 Notebook 02 — Agentic Medallion Transformation Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook2_medallion_architecture_flow.png)

---
### 📋 Notebook 02 Step-by-Step Execution Summary (100% Standalone — No Need to Run Notebook 01 First!)
1. **Step 1**: Configure parameterized `PROJECT_ID` (auto-detects active project if blank) & `LOCATION = "asia-southeast1"` (Singapore), enable the Data Engineering Agent APIs (`dataform.googleapis.com`, `cloudaicompanion.googleapis.com`), and stage all 8 compressed `.csv.gz` files in `gs://acsm-workshop-landing-${PROJECT_ID}/full_compressed/`.
2. **Step 0 (Bootstrap `acsm_bronze` Layer via `.sql` Scripts)**: Invoke `00_create_8_tables_ddl_with_descriptions.sql` and `01_load_data_from_gcs.sql` to create and load all **8 `acsm_bronze` tables** as native BigQuery `BASE TABLE`s so this notebook can be executed independently without running Notebook 01 first.
3. **Step 2 (`%%bigquery` SQL)**: Verify row counts across all **8 `acsm_bronze` base tables** (`1,398,284` rows).
4. **Step 3 (`%%bigquery` SQL)**: Create the target Medallion datasets (`acsm_silver` & `acsm_gold`) in `asia-southeast1`.
5. **Step 4 (UI Guide + Multi-Turn Natural-English Prompts)**: Copy-paste the **6 Incremental Natural-English Prompts** (Medallion DAG $\rightarrow$ Incremental Load $\rightarrow$ Dataform Assertions $\rightarrow$ BQML Model Training `acsm_silver.model_delinquency_propensity` $\rightarrow$ Batch BQML Scoring `acsm_gold.gold_aeon360_batch_ml_predictions` $\rightarrow$ Scheduled Workflow & Dataplex DQ Alerting) into the **BigQuery Data Engineering Agent (`+ Create` → `Pipeline`)**.
6. **Step 5 (`%%bigquery` SQL)**: Transparent SQL execution of the **4 Silver tables + 1 Gold Customer 360 table + 1 Silver BQML Model (`model_delinquency_propensity`) + 1 Gold Batch Predictions table (`gold_aeon360_batch_ml_predictions`)** directly inside this notebook.
7. **Step 6 (`%%bigquery` SQL)**: Verify `acsm_silver` & `acsm_gold` row counts, run the **Dual-Run Financial Control Total Reconciliation Audit (`0.00 MYR Variance`)**, and preview the batch ML predictions.
8. **Step 7 (Apache Airflow / Cloud Composer Orchestrator)**: Generate the production **Apache Airflow DAG (`acsm_medallion_dataform_orchestrator_dag.py`)** using Google Cloud's native Dataform operators (`DataformCreateCompilationResultOperator` $\rightarrow$ `DataformCreateWorkflowInvocationOperator` $\rightarrow$ `BigQueryCheckOperator`) and programmatically compile & invoke your Dataform Medallion DAG in `asia-southeast1`.

---
## Step 1: Configure Project & Region (`asia-southeast1`) and Enable Data Engineering Agent APIs
Run the cell below to set your `PROJECT_ID` and `LOCATION` (`asia-southeast1` Singapore), load the `%bigquery` SQL magic extension, and enable the required APIs (`dataform.googleapis.com` and `cloudaicompanion.googleapis.com`) for the **BigQuery Data Engineering Agent**.

In [ ]:
# @title 1. Set Parameterized `PROJECT_ID` (Auto-Detects Active Project if Blank), Singapore Region (`asia-southeast1`), Enable APIs & Stage All 8 `.csv.gz` Files in GCS
import os
import subprocess

PROJECT_ID = ""  # @param {type:"string"}
if not PROJECT_ID or PROJECT_ID.startswith("<"):
    PROJECT_ID = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
    )
LOCATION = "asia-southeast1"  # @param {type:"string"}
BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

# Load the native BigQuery SQL magic so all subsequent cells run pure SQL against $PROJECT_ID
%load_ext google.cloud.bigquery

# 1. Enable BigQuery Pipelines (Dataform), BigLake, Cloud Storage, and Gemini for Google Cloud (Data Engineering Agent)
!gcloud config set project $PROJECT_ID
!gcloud services enable storage.googleapis.com bigquery.googleapis.com biglake.googleapis.com dataform.googleapis.com cloudaicompanion.googleapis.com --project=$PROJECT_ID

# 2. Ensure all 8 compressed .csv.gz files are staged in gs://acsm-workshop-landing-${PROJECT_ID}/full_compressed/
#    so Notebook 02 can run 100% independently even if Notebook 01 was not run first!
![ -d aeon-credit-gcp-workshop ] || git clone https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git
!gcloud storage buckets describe gs://{BUCKET_NAME} >/dev/null 2>&1 || gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/

# 3. Provision service identities and grant BigQuery + Storage permissions to the Dataform & Gemini Service Agents
PROJECT_NUMBER = subprocess.check_output(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"], text=True
).strip()

!gcloud beta services identity create --service=dataform.googleapis.com --project=$PROJECT_ID 2>/dev/null || true
!gcloud beta services identity create --service=cloudaicompanion.googleapis.com --project=$PROJECT_ID 2>/dev/null || true

for sa in [
    f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-dataform.iam.gserviceaccount.com",
    f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-cloudaicompanion.iam.gserviceaccount.com",
    f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-bigqueryconnection.iam.gserviceaccount.com",
]:
    for role in [
        "roles/biglake.admin",
        "roles/storage.objectAdmin",
        "roles/bigquery.connectionUser",
        "roles/bigquery.dataEditor",
        "roles/bigquery.jobUser",
    ]:
        subprocess.run(
            ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID, f"--member={sa}", f"--role={role}", "--quiet"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

print(f"✅ Ready! Active Project={PROJECT_ID} (#{PROJECT_NUMBER}) | Region={LOCATION} (Singapore)")
print(f"✅ All 8 compressed .csv.gz files staged in gs://{BUCKET_NAME}/full_compressed/ for standalone execution!")

---
## Step 0 (Standalone Bronze Layer Bootstrap): Invoke `.sql` Scripts to Create & Load All 8 `acsm_bronze` Base Tables
Run the cell below to execute `00_create_8_tables_ddl_with_descriptions.sql` (DDL) and `01_load_data_from_gcs.sql` (DML) so all **8 `acsm_bronze` tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection`, `m3CIF`, and `dimProduct`) exist as native BigQuery `BASE TABLE`s in `asia-southeast1` — even if Notebook 01 was not run first.

In [ ]:
# @title 0.1 Invoke `00_create_8_tables_ddl_with_descriptions.sql` & `01_load_data_from_gcs.sql` to Bootstrap `acsm_bronze`
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/00_create_8_tables_ddl_with_descriptions.sql
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/01_load_data_from_gcs.sql
print("✅ Bronze layer (`acsm_bronze`) initialized with all 8 base tables!")

---
## Step 2: Pre-Flight Check — Verify the 8 Source Tables in `acsm_bronze` (`%%bigquery`)
Before creating the Silver and Gold Medallion layers, run the `%%bigquery` SQL cell below to verify that all **8 core ACSM tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection`, `m3CIF`, `dimProduct`) are loaded in `acsm_bronze`.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Verify all 8 source tables & views in acsm_bronze (6 Native Fact Tables + m3CIF Lakehouse Iceberg View + dimProduct AWS Glue Federated Iceberg View)
SELECT 'Fact_EP_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Judge`
UNION ALL
SELECT 'Fact_EP_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Sales`
UNION ALL
SELECT 'Fact_EP_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Collection`
UNION ALL
SELECT 'Fact_CC_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Judge`
UNION ALL
SELECT 'Fact_CC_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Sales`
UNION ALL
SELECT 'Fact_CC_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Collection`
UNION ALL
SELECT 'm3CIF'              AS table_name, 'GCP Lakehouse Iceberg View (GCS)'     AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.m3CIF`
UNION ALL
SELECT 'dimProduct'         AS table_name, 'AWS Glue Federated Iceberg View (S3)' AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.dimProduct`
ORDER BY table_name;

---
## Step 3: Create the Target Medallion Datasets (`acsm_silver` & `acsm_gold`) in Singapore (`asia-southeast1`)
The **BigQuery Data Engineering Agent** requires the target datasets (`acsm_silver` and `acsm_gold`) to exist in the same region (`asia-southeast1` Singapore) as `acsm_bronze` before generating and running the Dataform Medallion pipeline.

> 💡 **Note on BQML Model Creation**: Both the **BigQuery ML Delinquency Propensity Model (`acsm_silver.model_delinquency_propensity`)** and the downstream **Gold Batch Prediction Table (`acsm_gold.gold_aeon360_batch_ml_predictions`)** are created directly through **natural-language prompts in the Data Engineering Agent in Step 4** (and included in the transparent SQL execution in **Step 5**).

Run the `%%bigquery` SQL cell below to create both target datasets (`acsm_silver` and `acsm_gold`) (~2 seconds).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Create the Silver Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_silver`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Silver Medallion Layer: Standardized, type-cast, and deduplicated Customer CIF, EP/CC Underwriting, Collections tables, and BQML Delinquency Propensity Model in Singapore (asia-southeast1)'
);

-- 2. Create the Gold Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_gold`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Gold Medallion Layer: Unified AEON 360 Customer Risk, Affordability & Credit Exposure Feature Store and Batch ML Predictions in Singapore (asia-southeast1)'
);

-- 3. Verify all 3 Medallion Datasets (Bronze, Silver, Gold) in INFORMATION_SCHEMA
SELECT
  schema_name AS dataset_name,
  location,
  creation_time
FROM `region-asia-southeast1`.INFORMATION_SCHEMA.SCHEMATA
WHERE schema_name IN ('acsm_bronze', 'acsm_silver', 'acsm_gold')
ORDER BY schema_name;

---
## Step 4: How & Where to Copy-Paste the Medallion & BQML Prompts in the BigQuery Data Engineering Agent

Now that `acsm_bronze` (with **all 8 tables materialized for sampling and 100% table & column descriptions verified in Step 0**), `acsm_silver`, and `acsm_gold` are ready in **`asia-southeast1` (Singapore)**, follow these clicks in **BigQuery Studio** to generate and iteratively refine your **Medallion + Incremental Load + Dataform Assertions + BQML Model Training + Batch ML Scoring + SLA Alerting Pipeline DAG** using plain natural English.

### 🧭 Part A: Where to Click in BigQuery Studio
1. Open **Google Cloud Console** $\rightarrow$ navigate to **BigQuery** $\rightarrow$ **Studio**.
2. In the left **Explorer** pane, expand your project (`${PROJECT_ID}`) and verify you see the three datasets:
   - `acsm_bronze` *(contains all 8 source tables enriched with 100% `Mock Metadata.xlsx` table & column descriptions)*
   - `acsm_silver` *(created in Step 3)*
   - `acsm_gold` *(created in Step 3)*
3. At the top of the **BigQuery Studio workspace tab bar** (next to **`+ SQL query`** and **`+ Notebook`**), click **`+` (Create new)** $\rightarrow$ select **`Pipeline`**.
4. When prompted for the **Pipeline Location / Code Region**, select **`asia-southeast1 (Singapore)`**.
5. Inside the **Pipeline Canvas**, click the **Gemini Sparkle button (`Ask Data Engineering Agent` / `Generate with Gemini`)**.

---
### 🗣️ Part B (Interactive Multi-Turn Demo Flow — Recommended): Incremental Natural-English Prompts in Pipeline Builder
In a live customer demo, paste these prompts **incrementally (turn by turn)** into the **Data Engineering Agent** chat pane to showcase how Gemini iteratively evolves the pipeline from a base Medallion DAG into a production-grade, incremental, quality-gated, ML-training and batch-scoring workflow:

#### 🔹 Prompt 1 (Base Medallion Architecture: Bronze $\rightarrow$ Silver $\rightarrow$ Gold)
```text
Using the source tables in the `acsm_bronze` dataset, build a 3-tier Medallion pipeline into `acsm_silver` and `acsm_gold`:

1. In `acsm_silver`, create four cleansed domain tables:
   - `silver_customer_cif`: Deduplicate customer master records from `acsm_bronze.m3CIF` by customer ID (`CIF_ID`), keeping the most recent record based on `Rcd_DT`, with standardized demographic and income fields (`N_Age`, `B_NetIncome`, `B_AnnualIncome`, `State`, `Region`, `Occupation`).
   - `silver_ep_underwriting`: Cleanse Easy Payment loan applications from `acsm_bronze.Fact_EP_Judge`, standardize the customer ID and application date (`APPL_DT`), and retain key underwriting score, DSR, NDI, and financing amount metrics.
   - `silver_cc_underwriting`: Cleanse Credit Card applications from `acsm_bronze.Fact_CC_Judge`, standardize the application date (`Appl_DT`), and retain credit score, DSR, NDI, and approved credit limit metrics.
   - `silver_collections_summary`: Combine Easy Payment (`Fact_EP_Collection`) and Credit Card (`Fact_CC_Collection`) collections at the customer level to calculate total unpaid principal and worst collection score grade per customer.

2. In `acsm_gold`, create an executive Customer 360 table `acsm_gold.gold_aeon360_customer_profile` by joining `silver_customer_cif` with customer-level summaries from `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`, and active credit card usage from `acsm_bronze.dimProduct`.
```

#### 🔹 Prompt 2 (Follow-Up: Incremental Watermark Load Logic — `RFP C1.1.1.18, C1.1.6.6`)
```text
Build incremental load logic with timestamp filtering for daily underwriting and collection updates to reduce compute footprint during recurring pipeline executions.
```

#### 🔹 Prompt 3 (Follow-Up: Dataform Primary-Key & Non-Null Assertions — `RFP C1.1.1.6`)
```text
Add Dataform assertion tests for unique primary keys and non-null constraints across incremental silver tables to safeguard data integrity.
```

#### 🔹 Prompt 4 (Follow-Up: Create & Train BigQuery ML Delinquency Model in Dataform — `RFP C1.1.4.4`)
```text
In `acsm_silver`, train a BigQuery ML logistic regression model named `model_delinquency_propensity` as a Dataform operation with output enabled. Train the model on `acsm_gold.gold_aeon360_customer_profile` using customer age (`N_Age`), net income (`B_NetIncome`), annual income (`B_AnnualIncome`), state (`State`), region (`Region`), and occupation (`Occupation`) to predict a binary delinquency risk label (`delinquency_risk_flag`, set to 1 if `combined_unpaid_osp` is greater than 0, and 0 otherwise).
```
> 💡 **Important Before Prompt 5**: After the Data Engineering Agent generates `model_delinquency_propensity`, click **`Apply`** and **`Run`** once so BigQuery creates and trains `acsm_silver.model_delinquency_propensity`. Once the model exists in BigQuery, the Agent's dry-run validator will immediately pass when you add the batch prediction table in **Prompt 5**!

#### 🔹 Prompt 5 (Follow-Up: Add Batch BQML Delinquency Risk Scoring Table into ETL Dataform — `RFP C1.1.4.4`)
```text
In `acsm_gold`, create a batch ML prediction table named `acsm_gold.gold_aeon360_batch_ml_predictions` by running BigQuery ML batch inference (`ML.PREDICT`) with the trained model `acsm_silver.model_delinquency_propensity` over `acsm_gold.gold_aeon360_customer_profile` to score all customers with their predicted delinquency risk.
```

#### 🔹 Prompt 6 (Follow-Up: Scheduled Workflow Execution & Dataplex DQ SLA Alerting — `RFP C1.1.1.5, C1.1.1.7`)
```text
Set up scheduled workflow execution with Dataplex Data Quality monitoring on assertions to trigger automated alerting on SLA breaches.
```

---
### 📋 Part C (Alternative — 2-Phase Consolidated Natural-English Prompts)
Because BigQuery's dry-run validator requires `acsm_silver.model_delinquency_propensity` to exist in BigQuery before validating a downstream `ML.PREDICT` query, if you want to use consolidated prompts instead of 6 turns, run **Phase 1 (Build Medallion + Train BQML Model)** $\rightarrow$ click **`Run`**, and then paste **Phase 2 (Add Batch Prediction + Schedule)**:

#### **Phase 1 Consolidated Prompt (Medallion Tables + Assertions + BQML Model Creation)**
```text
Using the source tables in the `acsm_bronze` dataset, build a 3-tier Medallion pipeline and train a BigQuery ML delinquency model in `acsm_silver` and `acsm_gold`:

1. In `acsm_silver`, create four cleansed domain tables:
   - `silver_customer_cif`: Deduplicate customer master records from `acsm_bronze.m3CIF` by customer ID (`CIF_ID`), keeping the most recent record based on `Rcd_DT`, with standardized demographic and income fields (`N_Age`, `B_NetIncome`, `B_AnnualIncome`, `State`, `Region`, `Occupation`).
   - `silver_ep_underwriting`: Cleanse Easy Payment loan applications from `acsm_bronze.Fact_EP_Judge`, standardize the customer ID and application date (`APPL_DT`), and retain key underwriting score, DSR, NDI, and financing amount metrics.
   - `silver_cc_underwriting`: Cleanse Credit Card applications from `acsm_bronze.Fact_CC_Judge`, standardize the application date (`Appl_DT`), and retain credit score, DSR, NDI, and approved credit limit metrics.
   - `silver_collections_summary`: Combine Easy Payment (`Fact_EP_Collection`) and Credit Card (`Fact_CC_Collection`) collections at the customer level to calculate total unpaid principal and worst collection score grade per customer.

2. Build incremental load logic with timestamp filtering for daily underwriting and collection updates to reduce compute footprint during recurring pipeline executions.

3. Add Dataform assertion tests for unique primary keys and non-null constraints across incremental silver tables to safeguard data integrity.

4. In `acsm_gold`, create an executive Customer 360 table `acsm_gold.gold_aeon360_customer_profile` by joining `silver_customer_cif` with customer-level summaries from `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`, and active credit card usage from `acsm_bronze.dimProduct`.

5. In `acsm_silver`, train a BigQuery ML logistic regression model named `model_delinquency_propensity` as a Dataform operation with output enabled. Train the model on `acsm_gold.gold_aeon360_customer_profile` using customer age (`N_Age`), net income (`B_NetIncome`), annual income (`B_AnnualIncome`), state (`State`), region (`Region`), and occupation (`Occupation`) to predict a binary delinquency risk label (`delinquency_risk_flag`, set to 1 if `combined_unpaid_osp` is greater than 0, and 0 otherwise).
```

#### **Phase 2 Consolidated Prompt (Paste After Clicking `Run` on Phase 1)**
```text
In `acsm_gold`, create a batch ML prediction table named `acsm_gold.gold_aeon360_batch_ml_predictions` by running BigQuery ML batch inference (`ML.PREDICT`) with the trained model `acsm_silver.model_delinquency_propensity` over `acsm_gold.gold_aeon360_customer_profile` to score all customers with their predicted delinquency risk, and set up scheduled workflow execution with Dataplex Data Quality monitoring on assertions to trigger automated alerting on SLA breaches.
```

---
### ▶️ Part D: Review the Visual DAG & Run the Pipeline
1. The **Data Engineering Agent** will generate and refine your visual **Dataform Medallion + BQML DAG** on the Pipeline Canvas:
   - **4 Silver transformation nodes** (`silver_customer_cif`, `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`) with incremental logic & Dataform assertion checks (`uniqueKey`, `nonNull`).
   - **1 Gold Customer 360 join node** (`gold_aeon360_customer_profile`) waiting for all 4 Silver tables + `dimProduct`.
   - **1 Silver BQML Model Training node** (`model_delinquency_propensity`) training the Logistic Regression model over `gold_aeon360_customer_profile`.
   - **1 Gold Batch BQML Scoring node** (`gold_aeon360_batch_ml_predictions`) running `ML.PREDICT` over `gold_aeon360_customer_profile`.
2. Click any node on the canvas to inspect the generated SQL/SQLX definition and assertion rules.
3. Click **`Apply`** to accept the agent's generated nodes, then click **`Run`** at the top of the Pipeline Canvas (or run **Step 5** below to materialize the exact production schema for **Step 6** and **Notebook 03**).

---
## Step 5: Transparent `%%bigquery` SQL Medallion Execution (Direct SQL Option & Reconciliation)
In addition to (or as a direct SQL comparison to) running the Pipeline Canvas in Step 4, you can execute the **exact same 5 Silver & Gold Medallion table transformations + Dual-Run Financial Reconciliation Audit** right here in this notebook using `%%bigquery` SQL so every line of transformation logic is 100% transparent.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- 1. SILVER LAYER: acsm_silver.silver_customer_cif (Deduplicated Customer Master)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_customer_cif`
CLUSTER BY CIF_ID, State
OPTIONS (
  description = 'Governed Silver Customer Master (m3CIF) deduplicated by CIF_ID with typed income and PDPA consent flags.'
) AS
SELECT
  CAST(CIF_ID AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y-%m-%d', SUBSTR(CAST(Rcd_DT AS STRING), 1, 10)) AS record_refresh_date,
  TRIM(CAST(CIF_NM AS STRING)) AS CIF_NM,
  TRIM(CAST(Gender AS STRING)) AS Gender,
  TRIM(CAST(MaritalSts AS STRING)) AS MaritalSts,
  TRIM(CAST(Citizen AS STRING)) AS Citizen,
  TRIM(CAST(State AS STRING)) AS State,
  TRIM(CAST(Region AS STRING)) AS Region,
  TRIM(CAST(Race AS STRING)) AS Race,
  TRIM(CAST(Occupation AS STRING)) AS Occupation,
  CAST(EmpSts AS INT64) AS EmpSts,
  CAST(N_Age AS INT64) AS N_Age,
  CAST(N_YrStay AS NUMERIC) AS N_YrStay,
  CAST(N_YrJob AS NUMERIC) AS N_YrJob,
  CAST(B_NetIncome AS NUMERIC) AS B_NetIncome,
  CAST(B_GrossIncome AS NUMERIC) AS B_GrossIncome,
  CAST(B_AnnualIncome AS NUMERIC) AS B_AnnualIncome,
  COALESCE(TRIM(CAST(RecvPromo_FG AS STRING)), 'N') AS RecvPromo_FG
FROM `acsm_bronze.m3CIF`
QUALIFY ROW_NUMBER() OVER (PARTITION BY CAST(CIF_ID AS STRING) ORDER BY Rcd_DT DESC) = 1;

-- =============================================================================
-- 2. SILVER LAYER: acsm_silver.silver_ep_underwriting (Easy Payment Underwriting)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_ep_underwriting`
CLUSTER BY CIF_ID, APPL_STS
OPTIONS (
  description = 'Governed Silver Easy Payment (EP) Application & Underwriting Decisions from acsm_bronze.Fact_EP_Judge.'
) AS
SELECT
  CAST(APPL_NO AS STRING) AS APPL_NO,
  CAST(AGREE_NO AS STRING) AS AGREE_NO,
  CAST(CIF_NO AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y%m%d', NULLIF(TRIM(CAST(APPL_DT AS STRING)), '0')) AS application_date,
  TRIM(CAST(APPL_STS AS STRING)) AS APPL_STS,
  CAST(SCORING_POINT AS NUMERIC) AS SCORING_POINT,
  TRIM(CAST(SCORING_RANK AS STRING)) AS SCORING_RANK,
  TRIM(CAST(SCORE_DECISION AS STRING)) AS SCORE_DECISION,
  TRIM(CAST(LOAN_GRP AS STRING)) AS LOAN_GRP,
  CAST(FIN_AMT AS NUMERIC) AS FIN_AMT,
  CAST(INST_AMT AS NUMERIC) AS INST_AMT,
  CAST(INTEREST AS NUMERIC) AS INTEREST,
  CAST(TOTAL_INST AS INT64) AS TOTAL_INST,
  CAST(NetIncome AS NUMERIC) AS NetIncome,
  CAST(NDI AS NUMERIC) AS NDI,
  CAST(CUR_DSR AS NUMERIC) AS CUR_DSR,
  CAST(NEW_DSR AS NUMERIC) AS NEW_DSR,
  CAST(TOTAL_AEON_OSB AS NUMERIC) AS TOTAL_AEON_OSB
FROM `acsm_bronze.Fact_EP_Judge`;

-- =============================================================================
-- 3. SILVER LAYER: acsm_silver.silver_cc_underwriting (Credit Card Underwriting)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_cc_underwriting`
CLUSTER BY CIF_ID, ApplSts_ID
OPTIONS (
  description = 'Governed Silver Credit Card Application & Underwriting Decisions from acsm_bronze.Fact_CC_Judge.'
) AS
SELECT
  CAST(Appl_ID AS STRING) AS Appl_ID,
  CAST(Account_No AS STRING) AS Account_No,
  CAST(CIF_ID AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y%m%d', NULLIF(TRIM(CAST(Appl_DT AS STRING)), '0')) AS application_date,
  TRIM(CAST(ApplSts_ID AS STRING)) AS ApplSts_ID,
  TRIM(CAST(CardTyp_ID AS STRING)) AS CardTyp_ID,
  TRIM(CAST(CardBrand_ID AS STRING)) AS CardBrand_ID,
  TRIM(CAST(ScoreDecision_ID AS STRING)) AS ScoreDecision_ID,
  TRIM(CAST(ScoreRank_ID AS STRING)) AS ScoreRank_ID,
  CAST(NetIncome AS NUMERIC) AS NetIncome,
  CAST(NDI AS NUMERIC) AS NDI,
  CAST(CurrDSR AS NUMERIC) AS CurrDSR,
  CAST(NewDSR AS NUMERIC) AS NewDSR,
  CAST(B_CrLimit AS NUMERIC) AS B_CrLimit,
  CAST(Final_Score AS NUMERIC) AS Final_Score,
  TRIM(CAST(Final_ScoreDesc AS STRING)) AS Final_ScoreDesc
FROM `acsm_bronze.Fact_CC_Judge`;

-- =============================================================================
-- 4. SILVER LAYER: acsm_silver.silver_collections_summary (EP + CC Collections)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_collections_summary`
CLUSTER BY CIF_ID
OPTIONS (
  description = 'Customer-level Silver Collections & Delinquency Summary across Fact_EP_Collection and Fact_CC_Collection.'
) AS
WITH ep_col AS (
  SELECT
    CAST(CIF_No AS STRING) AS CIF_ID,
    SUM(CAST(Unpaid_OSP AS NUMERIC)) AS total_ep_unpaid_osp,
    MAX(TRIM(CAST(Score_Grade AS STRING))) AS ep_worst_grade
  FROM `acsm_bronze.Fact_EP_Collection`
  GROUP BY 1
),
cc_col AS (
  SELECT
    CAST(CIF_No AS STRING) AS CIF_ID,
    SUM(CAST(Unpaid_OSP AS NUMERIC)) AS total_cc_unpaid_osp,
    MAX(TRIM(CAST(Score_Grade AS STRING))) AS cc_worst_grade
  FROM `acsm_bronze.Fact_CC_Collection`
  GROUP BY 1
)
SELECT
  COALESCE(ep.CIF_ID, cc.CIF_ID) AS CIF_ID,
  COALESCE(ep.total_ep_unpaid_osp, 0) AS total_ep_unpaid_osp,
  COALESCE(cc.total_cc_unpaid_osp, 0) AS total_cc_unpaid_osp,
  COALESCE(ep.total_ep_unpaid_osp, 0) + COALESCE(cc.total_cc_unpaid_osp, 0) AS combined_unpaid_osp,
  GREATEST(COALESCE(ep.ep_worst_grade, 'A'), COALESCE(cc.cc_worst_grade, 'A')) AS worst_collection_score_grade
FROM ep_col ep
FULL OUTER JOIN cc_col cc
  ON ep.CIF_ID = cc.CIF_ID;

-- =============================================================================
-- 5. GOLD LAYER: acsm_gold.gold_aeon360_customer_profile (AEON 360 Feature Store)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_gold.gold_aeon360_customer_profile`
CLUSTER BY State, CIF_ID
OPTIONS (
  description = 'Gold AEON 360 Customer Risk, Affordability & Credit Exposure Feature Store joining CIF, EP Underwriting, CC Underwriting, Collections, and Card Utilization.'
) AS
WITH ep_agg AS (
  SELECT
    CIF_ID,
    COUNT(*) AS ep_app_count,
    ROUND(SUM(COALESCE(FIN_AMT, 0)), 2) AS total_ep_financed_myr,
    ROUND(AVG(NEW_DSR), 2) AS avg_ep_new_dsr
  FROM `acsm_silver.silver_ep_underwriting`
  GROUP BY CIF_ID
),
cc_agg AS (
  SELECT
    CIF_ID,
    COUNT(*) AS cc_app_count,
    ROUND(SUM(COALESCE(B_CrLimit, 0)), 2) AS total_cc_limit_myr,
    MAX(Final_Score) AS latest_ctos_score
  FROM `acsm_silver.silver_cc_underwriting`
  GROUP BY CIF_ID
),
card_agg AS (
  SELECT
    CAST(CIF_ID AS STRING) AS CIF_ID,
    COUNTIF(TRIM(CAST(Card_Status AS STRING)) = 'Active') AS active_card_count,
    ROUND(SUM(CAST(CP_CL_Usage AS NUMERIC)), 2) AS total_cp_usage_myr,
    ROUND(SUM(CAST(CP_CL_Available AS NUMERIC)), 2) AS total_cp_available_myr
  FROM `acsm_bronze.dimProduct`
  GROUP BY 1
)
SELECT
  c.CIF_ID,
  c.CIF_NM,
  c.State,
  c.Region,
  c.Occupation,
  c.N_Age,
  c.B_NetIncome,
  c.B_AnnualIncome,
  c.RecvPromo_FG,
  COALESCE(ep.ep_app_count, 0) AS ep_app_count,
  COALESCE(ep.total_ep_financed_myr, 0) AS total_ep_financed_myr,
  ep.avg_ep_new_dsr,
  COALESCE(cc.cc_app_count, 0) AS cc_app_count,
  COALESCE(cc.total_cc_limit_myr, 0) AS total_cc_limit_myr,
  cc.latest_ctos_score,
  COALESCE(col.total_ep_unpaid_osp, 0) AS total_ep_unpaid_osp,
  COALESCE(col.total_cc_unpaid_osp, 0) AS total_cc_unpaid_osp,
  COALESCE(col.combined_unpaid_osp, 0) AS combined_unpaid_osp,
  COALESCE(col.worst_collection_score_grade, 'NONE') AS worst_collection_score_grade,
  COALESCE(crd.active_card_count, 0) AS active_card_count,
  COALESCE(crd.total_cp_usage_myr, 0) AS total_cp_usage_myr,
  COALESCE(crd.total_cp_available_myr, 0) AS total_cp_available_myr
FROM `acsm_silver.silver_customer_cif` c
LEFT JOIN ep_agg ep USING (CIF_ID)
LEFT JOIN cc_agg cc USING (CIF_ID)
LEFT JOIN `acsm_silver.silver_collections_summary` col USING (CIF_ID)
LEFT JOIN card_agg crd USING (CIF_ID);

-- =============================================================================
-- 6. SILVER BQML MODEL NODE: acsm_silver.model_delinquency_propensity
--    (Matches Prompt 4: Dataform operation with output enabled)
-- =============================================================================
CREATE OR REPLACE MODEL `acsm_silver.model_delinquency_propensity`
OPTIONS (
  MODEL_TYPE = 'LOGISTIC_REG',
  INPUT_LABEL_COLS = ['delinquency_risk_flag'],
  AUTO_CLASS_WEIGHTS = TRUE,
  MAX_ITERATIONS = 5
) AS
SELECT
  N_Age,
  B_NetIncome,
  B_AnnualIncome,
  State,
  Region,
  Occupation,
  IF(COALESCE(combined_unpaid_osp, 0) > 0, 1, 0) AS delinquency_risk_flag
FROM `acsm_gold.gold_aeon360_customer_profile`;

-- =============================================================================
-- 7. GOLD LAYER (BATCH BQML INFERENCE NODE): acsm_gold.gold_aeon360_batch_ml_predictions
--    (Matches Prompt 5: Batch ML scoring over gold_aeon360_customer_profile)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_gold.gold_aeon360_batch_ml_predictions`
CLUSTER BY State, CIF_ID
OPTIONS (
  description = 'Gold Batch BQML Delinquency Risk Predictions scoring all 100,000 customers via ML.PREDICT(MODEL acsm_silver.model_delinquency_propensity).'
) AS
SELECT
  *
FROM ML.PREDICT(
  MODEL `acsm_silver.model_delinquency_propensity`,
  TABLE `acsm_gold.gold_aeon360_customer_profile`
);

---
## Step 6: Verify `acsm_silver` & `acsm_gold` Tables and Dual-Run Financial Reconciliation (`%%bigquery`)
Run the two `%%bigquery` SQL cells below to:
1. Inspect the created tables and row counts in `acsm_silver` and `acsm_gold`.
2. Run the **Dual-Run Financial Control Total Reconciliation Audit** (`Bronze vs. Silver/Gold`) to prove **0.00 MYR variance** across Easy Payment financed principal (`FIN_AMT`) and Collections unpaid principal (`Unpaid_OSP`).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Verify all created Medallion tables across acsm_silver and acsm_gold
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_silver.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
UNION ALL
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_gold.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
ORDER BY medallion_layer, table_name;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 2. Dual-Run Financial Control Total Reconciliation Audit (Bronze vs. Silver) + Preview Gold AEON 360
WITH checks AS (
  SELECT
    'Fact_EP_Judge -> silver_ep_underwriting' AS pipeline_flow,
    'FIN_AMT (Financed Principal MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_ep_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(FIN_AMT AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(FIN_AMT), 2) FROM `acsm_silver.silver_ep_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_CC_Judge -> silver_cc_underwriting' AS pipeline_flow,
    'B_CrLimit (Approved Credit Limit MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_cc_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(B_CrLimit AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(B_CrLimit), 2) FROM `acsm_silver.silver_cc_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_EP_Collection + Fact_CC_Collection -> silver_collections_summary' AS pipeline_flow,
    'Unpaid_OSP (Combined Unpaid Principal MYR)' AS control_metric,
    (SELECT COUNT(DISTINCT CIF_No) FROM (
      SELECT CIF_No FROM `acsm_bronze.Fact_EP_Collection`
      UNION DISTINCT
      SELECT CIF_No FROM `acsm_bronze.Fact_CC_Collection`
    )) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_collections_summary`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Collection`) +
      (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Collection`) AS bronze_total_myr,
    (SELECT ROUND(SUM(combined_unpaid_osp), 2) FROM `acsm_silver.silver_collections_summary`) AS silver_total_myr
)
SELECT
  pipeline_flow,
  control_metric,
  bronze_rows,
  silver_rows,
  bronze_total_myr,
  silver_total_myr,
  (silver_total_myr - bronze_total_myr) AS variance_myr,
  IF(bronze_rows = silver_rows AND ABS(silver_total_myr - bronze_total_myr) = 0, 'PASS (0.00 MYR VARIANCE)', 'INVESTIGATE') AS audit_status
FROM checks;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 3. Preview Top 10 Customers in the Gold Batch BQML Predictions Table (`acsm_gold.gold_aeon360_batch_ml_predictions`)
SELECT
  CIF_ID,
  CIF_NM,
  State,
  B_NetIncome,
  total_ep_financed_myr,
  total_cc_limit_myr,
  combined_unpaid_osp,
  worst_collection_score_grade,
  predicted_delinquency_risk_flag,
  ROUND(predicted_delinquency_risk_flag_probs[OFFSET(0)].prob, 4) AS predicted_delinquency_prob
FROM `acsm_gold.gold_aeon360_batch_ml_predictions`
ORDER BY total_ep_financed_myr + total_cc_limit_myr DESC
LIMIT 10;

---
## Step 7: Apache Airflow (Cloud Composer) Orchestrator to Invoke the Dataform Medallion DAG (`RFP C1.1.1.5, C1.1.1.6, C1.1.1.7`)

In enterprise production on Google Cloud, **Cloud Composer (Managed Apache Airflow)** acts as the cross-system control plane that schedules, compiles, and invokes the **BigQuery Dataform Medallion DAG** using official `apache-airflow-providers-google` operators:

| Airflow Task ID | Airflow Operator (`apache-airflow-providers-google`) | Responsibility in the ACSM Medallion Pipeline |
| :--- | :--- | :--- |
| **`1. verify_bronze_lakehouse_readiness`** | `BigQueryCheckOperator` | Verifies that all **8 `acsm_bronze` tables** (`1,398,284` rows across Native Fact tables + synced GCP/AWS Iceberg tables) are populated in `asia-southeast1`. |
| **`2. compile_dataform_medallion_repo`** | `DataformCreateCompilationResultOperator` | Compiles the Dataform SQLX repository/workspace in `asia-southeast1` and resolves the dependency graph (`acsm_bronze` $\rightarrow$ `acsm_silver` $\rightarrow$ Assertions $\rightarrow$ `acsm_gold`). |
| **`3. invoke_dataform_medallion_dag`** | `DataformCreateWorkflowInvocationOperator` | Executes the compiled Dataform DAG (`silver_customer_cif`, `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`, PK/Non-Null Assertions, `gold_aeon360_customer_profile`, and `gold_aeon360_batch_ml_predictions`) with `include_dependencies=True`. |
| **`4. audit_reconciliation_and_dq_sla`** | `BigQueryCheckOperator` | Validates **Zero Financial Variance (`0.00 MYR`)** between Bronze and Silver underwriting totals and triggers automated SLA alerting on breach. |

**Airflow Orchestration Topology:**
```text
[verify_bronze_lakehouse_readiness]
               │
               ▼
[compile_dataform_medallion_repo]  (DataformCreateCompilationResultOperator)
               │
               ▼
 [invoke_dataform_medallion_dag]   (DataformCreateWorkflowInvocationOperator)
               │
               ▼
 [audit_reconciliation_and_dq_sla] (BigQueryCheckOperator -> SLA Alert Callback)
```

Run the cell below to:
1. **Generate the production Apache Airflow DAG file** (`acsm_medallion_dataform_orchestrator_dag.py`) ready for Cloud Composer (`dags/` folder).
2. **Auto-discover your Dataform Repository & Workspace** in `asia-southeast1` and **execute the exact same 2-step Airflow orchestration sequence (`compile` $\rightarrow$ `createWorkflowInvocation`)** directly via the Dataform API so you can verify live DAG compilation and execution immediately!

In [ ]:
# @title 7.1 Generate Apache Airflow DAG (`acsm_medallion_dataform_orchestrator_dag.py`) & Trigger Dataform Medallion Workflow Invocation
import json
import os
import subprocess
import time
import urllib.request
import urllib.error

PROJECT_ID = os.environ.get("PROJECT_ID") or subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()
LOCATION = os.environ.get("LOCATION", "asia-southeast1")

# Optional overrides (if left blank, the script auto-discovers the latest Dataform repository & workspace in asia-southeast1)
DATAFORM_REPOSITORY_ID = ""  # @param {type:"string"}
DATAFORM_WORKSPACE_ID = ""  # @param {type:"string"}
EXECUTE_LIVE_DATAFORM_INVOCATION = True  # @param {type:"boolean"}

# -----------------------------------------------------------------------------
# 1. Auto-Discover Active Dataform Repository & Workspace in asia-southeast1
# -----------------------------------------------------------------------------
token = subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def call_dataform_api(url, method="GET", payload=None):
    data = json.dumps(payload).encode("utf-8") if payload is not None else None
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode("utf-8"))

base_url = f"https://dataform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{LOCATION}"

if not DATAFORM_REPOSITORY_ID:
    try:
        repos_resp = call_dataform_api(f"{base_url}/repositories")
        repos = repos_resp.get("repositories", [])
        if repos:
            # Pick the most recently created/available repository
            DATAFORM_REPOSITORY_ID = repos[-1]["name"].split("/")[-1]
            print(f"🔍 Auto-discovered Dataform Repository in {LOCATION}: {DATAFORM_REPOSITORY_ID}")
        else:
            DATAFORM_REPOSITORY_ID = "acsm-medallion-pipeline"
            print(f"ℹ️ No existing Dataform repository found in {LOCATION}; using default template ID: {DATAFORM_REPOSITORY_ID}")
    except Exception as e:
        DATAFORM_REPOSITORY_ID = "acsm-medallion-pipeline"
        print(f"ℹ️ Using default template Repository ID ({DATAFORM_REPOSITORY_ID}): {e}")

if not DATAFORM_WORKSPACE_ID:
    try:
        ws_resp = call_dataform_api(f"{base_url}/repositories/{DATAFORM_REPOSITORY_ID}/workspaces")
        workspaces = ws_resp.get("workspaces", [])
        if workspaces:
            DATAFORM_WORKSPACE_ID = workspaces[0]["name"].split("/")[-1]
            print(f"🔍 Auto-discovered Dataform Workspace: {DATAFORM_WORKSPACE_ID}")
        else:
            DATAFORM_WORKSPACE_ID = "default"
    except Exception:
        DATAFORM_WORKSPACE_ID = "default"

# -----------------------------------------------------------------------------
# 2. Write Production Apache Airflow (Cloud Composer) DAG File
# -----------------------------------------------------------------------------
airflow_dag_code = f'''"""
ACSM Track 1 — Apache Airflow (Cloud Composer) Orchestrator for BigQuery Dataform Medallion DAG
Region: {LOCATION} (Singapore)
RFP Clauses: C1.1.1.5 (Orchestration), C1.1.1.6 (Data Quality Assertions), C1.1.1.7 (SLA Alerting)
"""
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.google.cloud.operators.bigquery import BigQueryCheckOperator
from airflow.providers.google.cloud.operators.dataform import (
    DataformCreateCompilationResultOperator,
    DataformCreateWorkflowInvocationOperator,
)

PROJECT_ID = "{PROJECT_ID}"
REGION = "{LOCATION}"
REPOSITORY_ID = "{DATAFORM_REPOSITORY_ID}"
WORKSPACE_ID = "{DATAFORM_WORKSPACE_ID}"

def sla_breach_alert_callback(context):
    """Automated SLA & Data Quality breach notification hook (RFP Clause C1.1.1.7)."""
    task_id = context.get("task_instance").task_id
    dag_id = context.get("dag").dag_id
    exec_date = context.get("execution_date")
    print(f"[SLA ALERT] DAG={{dag_id}} | Task={{task_id}} breached SLA/DQ check at {{exec_date}}")

default_args = {{
    "owner": "acsm-data-engineering",
    "depends_on_past": False,
    "email_on_failure": True,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
    "on_failure_callback": sla_breach_alert_callback,
}}

with DAG(
    dag_id="acsm_medallion_dataform_orchestrator",
    description="Orchestrates ACSM Bronze -> Silver -> Gold + BQML Dataform DAG in Singapore (asia-southeast1)",
    default_args=default_args,
    schedule_interval="0 2 * * *",  # Daily at 02:00 AM MYT/SGT (UTC+8 adjusted)
    start_date=datetime(2026, 1, 1),
    catchup=False,
    max_active_runs=1,
    tags=["acsm", "medallion", "dataform", "bigquery", "bnm-rmit"],
) as dag:

    # 1. Pre-Flight Check: Ensure all 8 Bronze source tables are populated in asia-southeast1
    verify_bronze_lakehouse_readiness = BigQueryCheckOperator(
        task_id="verify_bronze_lakehouse_readiness",
        sql=f"""
            SELECT COUNT(*) = 8
            FROM `{PROJECT_ID}.acsm_bronze.INFORMATION_SCHEMA.TABLES`
            WHERE table_name IN (
                'm3CIF', 'dimProduct', 'Fact_EP_Judge', 'Fact_CC_Judge',
                'Fact_EP_Collection', 'Fact_CC_Collection', 'Cust_Bank_Acc_Txn', 'Fact_Wallet_Trans'
            )
        """,
        use_legacy_sql=False,
        location=REGION,
    )

    # 2. Compile the Dataform Repository (resolving Bronze -> Silver -> Gold -> BQML dependency graph)
    compile_dataform_medallion_repo = DataformCreateCompilationResultOperator(
        task_id="compile_dataform_medallion_repo",
        project_id=PROJECT_ID,
        region=REGION,
        repository_id=REPOSITORY_ID,
        compilation_result={{
            "git_commitish": "main",
            "code_compilation_config": {{
                "default_database": PROJECT_ID,
                "default_location": REGION,
            }},
        }},
    )

    # 3. Invoke the Compiled Dataform Workflow (Silver Incremental + Assertions + BQML Model + Gold Batch Scoring)
    invoke_dataform_medallion_dag = DataformCreateWorkflowInvocationOperator(
        task_id="invoke_dataform_medallion_dag",
        project_id=PROJECT_ID,
        region=REGION,
        repository_id=REPOSITORY_ID,
        workflow_invocation={{
            "compilation_result": "{{{{ task_instance.xcom_pull('compile_dataform_medallion_repo')['name'] }}}}",
            "invocation_config": {{
                "include_dependencies": True,
                "include_dependents": True,
                "fully_refresh_incremental_tables_enabled": False,
            }},
        }},
    )

    # 4. Post-Execution Financial Control-Total Reconciliation & Gold BQML Audit Gate
    audit_reconciliation_and_dq_sla = BigQueryCheckOperator(
        task_id="audit_reconciliation_and_dq_sla",
        sql=f"""
            SELECT
              ROUND(
                (SELECT SUM(CAST(FIN_AMT AS NUMERIC)) FROM `{PROJECT_ID}.acsm_bronze.Fact_EP_Judge`)
                - (SELECT SUM(CAST(FIN_AMT AS NUMERIC)) FROM `{PROJECT_ID}.acsm_silver.silver_ep_underwriting`),
                2
              ) = 0.00
              AND (SELECT COUNT(*) FROM `{PROJECT_ID}.acsm_gold.gold_aeon360_batch_ml_predictions`) > 0
        """,
        use_legacy_sql=False,
        location=REGION,
    )

    (
        verify_bronze_lakehouse_readiness
        >> compile_dataform_medallion_repo
        >> invoke_dataform_medallion_dag
        >> audit_reconciliation_and_dq_sla
    )
'''

dag_filename = "acsm_medallion_dataform_orchestrator_dag.py"
with open(dag_filename, "w", encoding="utf-8") as f:
    f.write(airflow_dag_code)

print(f"✅ Generated Apache Airflow DAG file: {os.path.abspath(dag_filename)}")
print(f"   • Target Project    : {PROJECT_ID}")
print(f"   • Target Region     : {LOCATION}")
print(f"   • Dataform Repo ID  : {DATAFORM_REPOSITORY_ID}")
print(f"   • Dataform Workspace: {DATAFORM_WORKSPACE_ID}")
print(
    f"   • To deploy to Cloud Composer: "
    f"gcloud composer environments storage dags import --environment=<COMPOSER_ENV> --location={LOCATION} --source={dag_filename}"
)

# -----------------------------------------------------------------------------
# 3. Execute the Airflow Dataform Operator Sequence Directly via Dataform API
#    (DataformCreateCompilationResult -> DataformCreateWorkflowInvocation)
# -----------------------------------------------------------------------------
if EXECUTE_LIVE_DATAFORM_INVOCATION:
    print("\n🚀 Executing Airflow Orchestration Sequence against Dataform API...")
    try:
        # Check if workspace exists; compile from workspace if present, otherwise gitCommitish="main"
        compile_payload = {
            "codeCompilationConfig": {
                "defaultDatabase": PROJECT_ID,
                "defaultLocation": LOCATION,
            }
        }
        try:
            ws_check = call_dataform_api(f"{base_url}/repositories/{DATAFORM_REPOSITORY_ID}/workspaces")
            if ws_check.get("workspaces"):
                ws_full_name = ws_check["workspaces"][0]["name"]
                compile_payload["workspace"] = ws_full_name
                print(f"   1️⃣ [DataformCreateCompilationResultOperator] Compiling workspace: {ws_full_name}")
            else:
                compile_payload["gitCommitish"] = "main"
                print(f"   1️⃣ [DataformCreateCompilationResultOperator] Compiling branch 'main' in {DATAFORM_REPOSITORY_ID}")
        except Exception:
            compile_payload["gitCommitish"] = "main"

        comp_res = call_dataform_api(
            f"{base_url}/repositories/{DATAFORM_REPOSITORY_ID}/compilationResults",
            method="POST",
            payload=compile_payload,
        )
        comp_name = comp_res.get("name")
        comp_errors = comp_res.get("compilationErrors", [])
        if comp_errors:
            print(f"   ⚠️ Compilation reported warnings/errors: {json.dumps(comp_errors, indent=2)}")
        else:
            print(f"   ✅ Compilation Result Created: {comp_name}")

        # Trigger Workflow Invocation (DataformCreateWorkflowInvocationOperator equivalent)
        print(f"   2️⃣ [DataformCreateWorkflowInvocationOperator] Triggering Dataform DAG execution...")
        inv_payload = {
            "compilationResult": comp_name,
            "invocationConfig": {
                "includeDependencies": True,
                "includeDependents": True,
                "fullyRefreshIncrementalTablesEnabled": False,
            },
        }
        inv_res = call_dataform_api(
            f"{base_url}/repositories/{DATAFORM_REPOSITORY_ID}/workflowInvocations",
            method="POST",
            payload=inv_payload,
        )
        inv_name = inv_res.get("name")
        inv_state = inv_res.get("state", "RUNNING")
        print(f"   ✅ Workflow Invocation Started: {inv_name} (Initial State: {inv_state})")

        # Poll briefly for execution status
        for _ in range(6):
            time.sleep(5)
            status_res = call_dataform_api(f"https://dataform.googleapis.com/v1beta1/{inv_name}")
            inv_state = status_res.get("state", "UNKNOWN")
            print(f"      ⏳ Dataform DAG Status: {inv_state}")
            if inv_state in ("SUCCEEDED", "FAILED", "CANCELLED"):
                break
        print(f"   🎯 Final Observed Dataform Workflow State: {inv_state}")
    except urllib.error.HTTPError as http_err:
        err_body = http_err.read().decode("utf-8", errors="ignore")
        print(
            f"   ℹ️ Live Dataform API invocation skipped or repo not yet created in Step 4 "
            f"(HTTP {http_err.code}): {err_body[:300]}"
        )
        print("   💡 Tip: Once you create and save your Pipeline in Step 4, re-run this cell to trigger it via the Dataform API!")
    except Exception as ex:
        print(f"   ℹ️ Live Dataform API invocation note: {ex}")